# Eureka · Entrenar la voz de Marc (gratis, en Colab)

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**.

Después: **Entorno de ejecución → Ejecutar todo**. El paso 3 te pedirá subir los dos archivos `voz_marc_parte1.zip` y `voz_marc_parte2.zip` (están en tu carpeta `EUREKA/CLON VOZ`).

Tarda en total **unos 45–60 minutos**. Deja la pestaña abierta. Al final se descarga solo un archivo `voz_marc_adaptador.zip`: déjalo en `EUREKA/CLON VOZ` y avisa a Claude.

Modelo: Chatterbox Multilingual (licencia MIT, uso comercial permitido) + LoRA entrenado con tus grabaciones.

In [ ]:
# 1. Comprobar la GPU
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU: cambia el entorno a T4')

In [ ]:
# 2. Instalar el entrenador (unos 5 minutos)
# Colab ya usa Python 3.13: instalamos el paquete sin la dependencia rusa (solo sirve para ruso)
# y aprovechamos el PyTorch que ya trae Colab.
!git clone -q https://github.com/Ahmed-Ezzat20/chatterbox-finetuning-multilingual.git cbft
%cd cbft
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q "setuptools<81" librosa==0.11.0 s3tokenizer "onnx>=1.18.0,<1.19.0" transformers==4.46.3 diffusers==0.29.0 resemble-perth==1.0.1 conformer==0.3.2 safetensors==0.5.3 soundfile pandas matplotlib 2>&1 | grep -i -E "^error" | head -10
!python -c "import torch; from chatterbox.mtl_tts import ChatterboxMultilingualTTS; print('INSTALACIÓN OK · torch', torch.__version__, '· GPU:', torch.cuda.is_available())"


In [ ]:
# 3. Subir tus grabaciones (elige los DOS zip a la vez)
from google.colab import files
up = files.upload()
for n in up: print('subido', n)
!unzip -oq voz_marc_parte1.zip && unzip -oq voz_marc_parte2.zip
import pandas as pd, os
df = pd.read_csv('audio_data/metadata.csv'); ok = df.file_name.map(lambda f: os.path.exists('audio_data/' + f)).sum()
print(len(df), 'frases,', ok, 'audios encontrados,', round(df.duration_seconds.sum()/60, 1), 'minutos')

In [ ]:
# 4. Ajustes: español de España, entrenamiento corto pensado para ~20 minutos de voz
import re
s = open('lora.py').read()
s = s.replace("language_id='ar'", "language_id='es'").replace('language_id: str = "ar"', 'language_id: str = "es"')
s = re.sub(r'^EPOCHS = .*$', 'EPOCHS = 12', s, flags=re.M)
s = re.sub(r'^WARMUP_STEPS = .*$', 'WARMUP_STEPS = 100', s, flags=re.M)
s = re.sub(r'^SAVE_EVERY_N_STEPS = .*$', 'SAVE_EVERY_N_STEPS = 1000', s, flags=re.M)
s = re.sub(r'^LEARNING_RATE = .*$', 'LEARNING_RATE = 5e-5', s, flags=re.M)
open('lora.py', 'w').write(s)
print([l for l in s.splitlines() if l.startswith(('EPOCHS','LEARNING_RATE','BATCH_SIZE','LORA_RANK'))], "language_id='es'" in s)

In [ ]:
# 5. Entrenar (30-50 minutos). Verás cómo baja la pérdida (loss).
!python lora.py 2>&1 | grep -v -i warning | tail -n 60
import os
if not os.path.exists('checkpoints_lora/final_lora_adapter.pt'):
    raise SystemExit('EL ENTRENAMIENTO HA FALLADO: haz captura de las últimas líneas de arriba y mándasela a Claude')
print('ENTRENAMIENTO OK')


In [ ]:
# 6. Prueba rápida con tu voz entrenada (se ejecuta aparte para evitar líos de versiones)
open('prueba.py','w').write('''
import soundfile as sf, sys, glob
sys.path.insert(0, ".")
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from lora import load_lora_adapter
m = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
load_lora_adapter(m, "checkpoints_lora/final_lora_adapter.pt", device="cuda")
ref = sorted(glob.glob("audio_data/audio/*"))[0]
w = m.generate("¿Sabías que el pulpo tiene tres corazones? Y no es lo más raro que tiene.", language_id="es", audio_prompt_path=ref)
sf.write("prueba_voz_marc.wav", w.squeeze().cpu().numpy(), m.sr)
print("PRUEBA OK")
''')
!python prueba.py 2>&1 | grep -v -i warning | tail -n 30
from IPython.display import Audio; Audio('prueba_voz_marc.wav')


In [ ]:
# 7. Descargar el adaptador (déjalo en EUREKA/CLON VOZ)
!zip -qj voz_marc_adaptador.zip checkpoints_lora/final_lora_adapter.pt prueba_voz_marc.wav training_metrics.png 2>/dev/null; ls -la voz_marc_adaptador.zip
files.download('voz_marc_adaptador.zip')